<a href="https://colab.research.google.com/github/MDMynulHasan/Machine-Learning-Accelerated-Screening/blob/main/A2BB'H6_family_screening_by_ML_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Installing Clone of the MD-HIT from github for eliminating the redundancy

In [ ]:
# 1. Installing required scientific libraries
!pip install pymatgen mendeleev pandas numpy scikit-learn

# 2. Clone for the MD-HIT repository from GitHub
!git clone https://github.com/usccolumbia/MD-HIT.git

# 3. Verifying the file is uploaded (Optional check)
import os
if not os.path.exists("A2BBpH6_hypothetical_features.csv"):
    print("⚠️ WARNING: Please upload your 'A2BBpH6_hypothetical_features.csv' file to the Colab files section on the left!")
else:
    print("✅ File found. Ready to proceed.")

This Library requires for clone

In [ ]:
# 1. Clone the ElMD repository
!git clone https://github.com/lrcfmd/ElMD.git

# 2. Install it so Python can find it
!pip install ./ElMD

avoiding redundancy for hypothetical A2BB'H6

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import euclidean_distances
from sklearn.preprocessing import StandardScaler

# 1. Loading my Data
# ---------------------------------------------------------
filename = "A2BBpH6_hypothetical_features.csv"
df = pd.read_csv(filename)
print(f"Loaded {len(df)} materials.") #Printing the length of the original hypothetical csv file

# 2. Prepare the 'Chemical Space'

# selected all columns that contain 'MagpieData'.
# These numbers describe the chemistry (electronegativity, radii, etc.)
feature_cols = [c for c in df.columns if "MagpieData" in c]
X = df[feature_cols].values

#  'scale' the data so big numbers (like Melting Temp) don't
# overshadow small numbers (like Electronegativity).
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 3. The MD-HIT Logic (Greedy Redundancy Removal)

# We define a 'threshold'. If two materials are closer than this,
# we throw away the second one.
# Threshold 1.0 is a good starting point for scaled Euclidean distance.
threshold = 1.0

kept_indices = []
# We start with all indices as candidates
remaining_indices = list(range(len(df)))

print("Starting screening... (this may take a moment)")

while remaining_indices:
    # Pick the first available material in the list
    current_idx = remaining_indices.pop(0)
    kept_indices.append(current_idx)

    # If no one else is left, we are done
    if not remaining_indices:
        break

    # Calculate distance from this material to ALL remaining materials
    # (This replaces the broken ElMD calculation)
    current_feat = X_scaled[current_idx].reshape(1, -1)
    remaining_feats = X_scaled[remaining_indices]

    dists = euclidean_distances(current_feat, remaining_feats).flatten()

    # Find which materials are "different enough" (distance > threshold)
    # We keep only those. The ones closer than threshold are discarded.
    mask_keep = dists > threshold

    # Update the remaining list to only include the distinct ones
    remaining_indices = [remaining_indices[i] for i in range(len(remaining_indices)) if mask_keep[i]]

# 4. Saving the Result
# ---------------------------------------------------------
df_screened = df.iloc[kept_indices]
output_file = "A2BBpH6_screened_native.csv"
df_screened.to_csv(output_file, index=False)

print("\n" + "="*40)
print(f"Screening Complete!")
print(f"Original Candidates: {len(df)}")
print(f"Selected Candidates: {len(df_screened)}")
print(f"Removed: {len(df) - len(df_screened)} (Too similar)")
print(f"Saved to: {output_file}")
print("="*40)

CELL 1: SETUP, TRAINING & PREDICTION

In [ ]:

!pip install -q matminer scikit-learn pandas numpy pymatgen==2023.11.12 matplotlib seaborn shap



In [ ]:
# Install compatible versions for ALL libraries
# We force numpy < 2.0 (for matminer) AND shap < 0.50 (to match numpy)
!pip install "numpy<2.0" "shap<0.45.0" --force-reinstall
!pip install matminer scikit-learn pandas "pymatgen==2023.11.12" matplotlib seaborn

Download OQMD DATA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from matminer.datasets import load_dataset
from matminer.featurizers.composition import ElementProperty
from pymatgen.core import Composition
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

Training data by Hist Gradient Boosting Regressor

DUE to the language difference we only load the material project data

In [ ]:


# 1. Loading MATERIALS PROJECT DATA (Clean & Consistent)

print("⬇️  Loading Materials Project Data...")
# This dataset is the "Gold Standard" for formation energy
df_mp = load_dataset("matbench_mp_e_form")

# Fix the structure/composition format
print(" Extracting chemical formulas...")
df_mp['composition'] = df_mp['structure'].apply(lambda x: x.composition)

# We use the FULL dataset (~132,000 materials).
# MP is cleaner than OQMD, so we don't need to sample down for safety.
print(f" Training Data: {len(df_mp)} verified materials.")


# 2. FEATURIZE (Convert Chemistry -> Math)

print("  Featurizing (This takes times)...")
ep_feat = ElementProperty.from_preset(preset_name="magpie")

# Featurize Training Data
X_raw = ep_feat.featurize_dataframe(df_mp, col_id="composition", ignore_errors=True)
features = ep_feat.feature_labels()
X = X_raw[features]
y = X_raw["e_form"]


# 3. TRAIN & VALIDATE

print("Training High-Accuracy Model...")
# Split: 90% Train, 10% Test
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.1, random_state=42)

model = HistGradientBoostingRegressor(random_state=42, max_iter=300)
model.fit(X_tr, y_tr)

# Check Performance
y_pred_test = model.predict(X_te)
mae = mean_absolute_error(y_te, y_pred_test)
r2 = r2_score(y_te, y_pred_test)

print("\n" + "="*40)
print(f"🏆 FINAL MODEL ACCURACY (MP Only)")
print("="*40)
print(f"MAE: {mae:.4f} eV/atom")
print(f"R²:  {r2:.4f}")
print("="*40)

if mae < 0.05:
    print("✅ STATUS: EXCELLENT. Ready for screening.")
else:
    print("⚠️ STATUS: GOOD, but check if Magpie features loaded correctly.")

# ==========================================
# 4. PREDICT YOUR CANDIDATES
# ==========================================
filename = "A2BBpH6_screened_native.csv"
print(f"🔮 Predicting stability for {filename}...")

try:
    df_user = pd.read_csv(filename)
    df_user = df_user[['formula']].drop_duplicates()
    df_user['composition'] = df_user['formula'].apply(Composition)

    # Featurize your candidates
    X_user_raw = ep_feat.featurize_dataframe(df_user, col_id="composition", ignore_errors=True)
    X_user = X_user_raw[features]

    # Predict
    df_user['Predicted_E_form'] = model.predict(X_user)

    # Define Stability (Threshold: -0.1 eV/atom is safer for ML)
    df_user['Stability'] = ['Stable' if x < -0.05 else 'Unstable' for x in df_user['Predicted_E_form']]

    # Save Results
    output_file = "A2BBpH6_Final_Predictions_MP_Only.csv"
    df_user.to_csv(output_file, index=False)
    print(f"✅ SUCCESS! Results saved to '{output_file}'")

    # ==========================================
    # 5. VISUALIZATION
    # ==========================================
    plt.figure(figsize=(12, 5))

    # Plot 1: Parity Plot (How good is the model?)
    plt.subplot(1, 2, 1)
    plt.hexbin(y_te, y_pred_test, gridsize=40, cmap='Blues', mincnt=1)
    plt.plot([y_te.min(), y_te.max()], [y_te.min(), y_te.max()], 'r--', lw=2)
    plt.title(f"Model Accuracy (MAE: {mae:.3f} eV/atom)")
    plt.xlabel("Actual DFT Energy (eV/atom)")
    plt.ylabel("ML Predicted Energy (eV/atom)")

    # Plot 2: Your Candidates
    plt.subplot(1, 2, 2)
    sns.histplot(data=df_user, x="Predicted_E_form", hue="Stability",
                 palette={"Stable": "green", "Unstable": "red"}, kde=True)
    plt.axvline(-0.05, color='k', linestyle='--', label="Stability Threshold")
    plt.title("Stability of Your A2BB'H6 Candidates")
    plt.xlabel("Predicted Formation Energy (eV/atom)")

    plt.tight_layout()
    plt.show()

    # Show Top 5

    print("\n🌟 TOP 5 MOST STABLE CANDIDATES:")
    print(df_user[df_user['Stability']=='Stable'].sort_values('Predicted_E_form').head(5))

except Exception as e:
    print(f"❌ Error with candidate file: {e}")
    print("Make sure 'A2BBpH6_screened_native.csv' is uploaded.")

In [ ]:
# ============================================================================
# CHECKPOINT SAVE — run ONCE after Cell 16, then never again
# ----------------------------------------------------------------------------
# Pickles every object the W1/W2/W4 validation cells will need, so you can
# skip the 30-minute matbench featurization on future Colab sessions.
# ============================================================================
import pickle, os

checkpoint = {
    "df_mp":       df_mp,        # has 'composition' column
    "X":           X,            # full Magpie feature matrix
    "y":           y,            # formation energies
    "X_tr":        X_tr, "X_te": X_te,
    "y_tr":        y_tr, "y_te": y_te,
    "y_pred_test": y_pred_test,
    "features":    features,
}
# ep_feat and model are NOT pickled — too large and not strictly needed for W1/W2/W4
with open("ml_checkpoint.pkl", "wb") as f:
    pickle.dump(checkpoint, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f"Saved checkpoint: {os.path.getsize('ml_checkpoint.pkl')/1e6:.1f} MB")
print("Download this file and re-upload it in future Colab sessions to skip Cell 16.")

In [ ]:
# ============================================================================
# CHECKPOINT RESTORE — replaces Cell 16 on subsequent sessions
# ----------------------------------------------------------------------------
# Requires ml_checkpoint.pkl in /content/. Upload via the Files panel.
# ============================================================================
import pickle
with open("ml_checkpoint.pkl", "rb") as f:
    ck = pickle.load(f)
df_mp       = ck["df_mp"]
X           = ck["X"];        y     = ck["y"]
X_tr        = ck["X_tr"];     X_te  = ck["X_te"]
y_tr        = ck["y_tr"];     y_te  = ck["y_te"]
y_pred_test = ck["y_pred_test"]
features    = ck["features"]
print(f"Restored: df_mp={len(df_mp)}, X_te={len(X_te)}, features={len(features)}")

Since the model show mae  0.15 so filter outing the materials

In [ ]:
import pandas as pd

# 1. Load your prediction results
df = pd.read_csv("A2BBpH6_Final_Predictions_MP_Only.csv")
print(f"Total Candidates Scanned: {len(df)}")

# 2. Apply the "Reviewer-Safe" Filter
# We use -0.15 eV (your Model Error) as the safety buffer.
# If the model says -0.20, and it's wrong by +0.15, it's still -0.05 (Stable).
safe_threshold = -0.15

df_golden = df[df['Predicted_E_form'] < safe_threshold].copy()
df_golden = df_golden.sort_values('Predicted_E_form')

print(f"Candidates passing the Safety Filter (< {safe_threshold} eV): {len(df_golden)}")

# 3. Save the "Golden List" for your paper
df_golden.to_csv("A2BBpH6_Golden_Candidates.csv", index=False)
print("✅ Saved 'A2BBpH6_Golden_Candidates.csv'. These are your winners.")

# Show the top 10
print(df_golden[['formula', 'Predicted_E_form', 'Stability']].head(10))

CELL: MODEL PERFORMANCE METRICS

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# 1. CALCULATE METRICS

# Make predictions on the test set (X_te) created in the previous training step
print("📊 Calculating Model Performance Metrics...")
y_pred_test = model.predict(X_te)

# Calculate standard regression metrics
mae = mean_absolute_error(y_te, y_pred_test)
rmse = np.sqrt(mean_squared_error(y_te, y_pred_test))
r2 = r2_score(y_te, y_pred_test)

# Print the "Scorecard" for your paper
print("\n" + "="*50)
print("MODEL PERFORMANCE REPORT (Materials Project Only)")
print("="*50)
print(f"Training Data Size:        {len(X_tr)} materials")
print(f"Test Data Size:            {len(X_te)} materials")
print("-" * 50)
print(f"Mean Absolute Error (MAE): {mae:.4f} eV/atom")
print(f"Root Mean Sq Error (RMSE): {rmse:.4f} eV/atom")
print(f"R² Score (Fit Quality):    {r2:.4f}")
print("-" * 50)

# Interpretation
if r2 > 0.90:
    print("✅ R² > 0.90: Excellent fit. The model captures the physics well.")
elif r2 > 0.80:
    print("✅ R² > 0.80: Good fit. Suitable for screening.")
else:
    print("⚠️ R² < 0.80: The model might be struggling with complex chemistry.")

# ==========================================
# 2. GENERATE PUBLICATION PLOTS
# ==========================================
plt.figure(figsize=(14, 6))

# --- PLOT 1: PARITY PLOT (Actual vs. Predicted) ---
plt.subplot(1, 2, 1)
# Hexbin plot is better than scatter for 130k points (shows density)
hb = plt.hexbin(y_te, y_pred_test, gridsize=50, cmap='Blues', mincnt=1, bins='log')
cb = plt.colorbar(hb, label='Log(Count)')

# Draw the "Perfect Prediction" line (y=x)
min_val = min(y_te.min(), y_pred_test.min())
max_val = max(y_te.max(), y_pred_test.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Ideal Fit (y=x)')

plt.title(f"Parity Plot ($R^2$={r2:.3f})", fontsize=14)
plt.xlabel("Actual DFT Energy (eV/atom)", fontsize=12)
plt.ylabel("ML Predicted Energy (eV/atom)", fontsize=12)
plt.legend()
plt.grid(True, alpha=0.2)

# --- PLOT 2: RESIDUAL HISTOGRAM (Error Distribution) ---
plt.subplot(1, 2, 2)
residuals = y_te - y_pred_test
sns.histplot(residuals, bins=50, color='purple', kde=True, edgecolor='black', alpha=0.7)
plt.axvline(0, color='k', linestyle='--', linewidth=1)

# Add text box with MAE/RMSE
stats_text = f"MAE: {mae:.3f} eV/atom\nRMSE: {rmse:.3f} eV/atom"
plt.gca().text(0.05, 0.95, stats_text, transform=plt.gca().transAxes,
               fontsize=12, verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

plt.title("Error Distribution (Residuals)", fontsize=14)
plt.xlabel("Prediction Error (eV/atom)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# 5. VISUALIZATION
# ==========================================
plt.figure(figsize=(10, 6))
sns.histplot(data=df_user, x="Predicted_E_form", hue="Stability",
             palette={"Stable": "green", "Unstable": "red"}, kde=True)
plt.axvline(-0.05, color='k', linestyle='--', label="Stability Threshold")
plt.title(f"Stability Prediction (Trained on MP + OQMD)\nMAE: {mae:.3f} eV/atom")
plt.xlabel("Formation Energy (eV/atom)")
plt.legend()
plt.show()

# Show the best ones
print("\n🏆 TOP 5 MOST STABLE CANDIDATES:")
print(df_user[df_user['Stability']=='Stable'].sort_values('Predicted_E_form').head(5))

In [ ]:
import shap
import matplotlib.pyplot as plt

# ==========================================
# 1. SETUP SHAP (The Fast Way)
# ==========================================
print("🕵️‍♀️ Calculating SHAP Values (Feature Importance)...")

# CRITICAL STEP: Sample only 500 points for the explanation
# This makes it run in < 1 minute instead of 3 hours.
X_shap_sample = X_te.sample(n=500, random_state=42)

# Create the Explainer
# Since HistGradientBoosting is complex, we use the generic Explainer with the sample
explainer = shap.Explainer(model.predict, X_shap_sample)

# Calculate SHAP values (This might take ~30-60 seconds)
shap_values = explainer(X_shap_sample)

print("✅ SHAP calculation complete!")

# ==========================================
# 2. GENERATE THE PLOT
# ==========================================
plt.figure(figsize=(10, 6))

# The "Bee Swarm" plot - The Gold Standard for Q1 Journals
shap.summary_plot(shap_values, X_shap_sample, show=False)

plt.title("Feature Importance (What drives Stability?)", fontsize=14)
plt.tight_layout()
plt.show()

# ==========================================
# 3. INTERPRETATION FOR YOUR PAPER
# ==========================================
print("\n" + "="*40)
print("HOW TO READ THIS PLOT:")
print("="*40)
print("1. RED dots = High value of that feature.")
print("2. BLUE dots = Low value of that feature.")
print("3. RIGHT side = Makes energy HIGHER (More Unstable).")
print("4. LEFT side = Makes energy LOWER (More Stable).")
print("-" * 40)
print("EXAMPLE: If 'Magpie Mean Electronegativity' has BLUE dots on the LEFT:")
print("It means 'Low Electronegativity makes the material MORE STABLE'.")

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

# ==========================================
# 1. LOAD & PARSE DATA
# ==========================================
print("📂 Loading results...")
df = pd.read_csv("A2BBpH6_Final_Predictions_MP_Only.csv")

# Function to extract elements from "Rb2 Mn1 Ni1 H6" style composition
def parse_elements(comp_str):
    parts = comp_str.split()
    A, B, B_prime = None, None, None
    bs = []

    for p in parts:
        element = re.sub(r'[0-9]+', '', p) # Remove numbers
        count = re.sub(r'[A-Za-z]+', '', p) # Remove letters

        if element == 'H': continue
        if count == '2':
            A = element
        elif count == '1':
            bs.append(element)

    # Sort B and B' alphabetically so (Ti, Ni) is same as (Ni, Ti) for the plot
    bs.sort()
    if len(bs) >= 2:
        return A, bs[0], bs[1]
    else:
        return None, None, None

# Apply parsing
print("⚗️  Extracting A, B, and B' elements...")
df[['A_site', 'B_site', 'B_prime_site']] = df['composition'].apply(
    lambda x: pd.Series(parse_elements(x))
)

# Drop any rows that failed parsing
df_clean = df.dropna(subset=['A_site', 'B_site', 'B_prime_site'])

# ==========================================
# 2. PLOT 1: THE HEATMAP (B vs B')
# ==========================================
print("🎨 Generating Heatmap...")
# Pivot table: Average energy for every B-B' pair
heatmap_data = df_clean.pivot_table(index='B_site', columns='B_prime_site',
                                    values='Predicted_E_form', aggfunc='mean')

plt.figure(figsize=(12, 10))
sns.heatmap(heatmap_data, cmap='RdYlGn_r', annot=False, linewidths=0.5,
            cbar_kws={'label': 'Formation Energy (eV/atom)'})
plt.title("Stability Heatmap: Interaction of Transition Metals (B vs B')", fontsize=15)
plt.xlabel("B' Site Element", fontsize=12)
plt.ylabel("B Site Element", fontsize=12)
plt.tight_layout()
plt.show()

# ==========================================
# 3. PLOT 2: VIOLIN PLOT (Effect of A-Site)
# ==========================================
print("🎻 Generating Violin Plot...")
# Sort A-sites by atomic radius (roughly) or groups for cleaner plot
# We just sort by mean energy to show the "Best to Worst"
order = df_clean.groupby('A_site')['Predicted_E_form'].median().sort_values().index

plt.figure(figsize=(12, 6))
sns.violinplot(data=df_clean, x='A_site', y='Predicted_E_form', order=order, palette='viridis')
plt.axhline(-0.15, color='r', linestyle='--', label='Stability Threshold')

plt.title("Effect of A-Site Cation on Stability", fontsize=15)
plt.xlabel("A-Site Cation", fontsize=12)
plt.ylabel("Formation Energy (eV/atom)", fontsize=12)
plt.legend()
plt.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

# ==========================================
# 1. USER INPUT (CRITICAL STEP)
# ==========================================
# Look at your DFT results in the paper.
# What is the Formation Energy of K2LiScH6 in eV/atom?
# If you only have Desorption Enthalpy (49 kJ/mol), estimate E_form approx -0.3 to -0.4 eV/atom.
K2LiScH6_ENERGY = -0.328547423  # <--- REPLACE THIS WITH YOUR EXACT DFT VALUE!

print(f"o Highlighting K2LiScH6 at {K2LiScH6_ENERGY} eV/atom")

# ==========================================
# 2. LOAD & PREPARE DATA
# ==========================================
df = pd.read_csv("A2BBpH6_Final_Predictions_MP_Only.csv")

# Define Ionic Radii (Shannon Radii in Angstroms)
radii = {
    'Li': 0.76, 'Mg': 0.72, 'Na': 1.02, 'Ca': 1.00,
    'Sc': 0.745, 'Ti': 0.605, 'V': 0.54, 'Cr': 0.615,
    'Mn': 0.83, 'Fe': 0.61, 'Co': 0.65, 'Ni': 0.69,
    'Cu': 0.73, 'Zn': 0.74, 'Al': 0.535, 'K': 1.38,
    'Rb': 1.52, 'Cs': 1.67
}

# Parse A-site again
def get_A_site(formula):
    # Simple regex to find the first element (A-site)
    match = re.match(r"([A-Z][a-z]?)", formula)
    return match.group(1) if match else None

df['A_site'] = df['formula'].apply(get_A_site)
df['Radius'] = df['A_site'].map(radii)

# ==========================================
# 3. PLOT 1: THE "GOLDILOCKS" SCATTER PLOT
# ==========================================
plt.figure(figsize=(12, 7))

# Plot the ML Candidates (Background context)
sns.scatterplot(data=df, x='Radius', y='Predicted_E_form',
                hue='A_site', palette='viridis', alpha=0.6, s=80, edgecolor='k')

# Highlight K2LiScH6 (The Star)
# K radius = 1.38 A
plt.scatter([1.38], [K2LiScH6_ENERGY], color='red', s=400, marker='o',
            edgecolor='black', zorder=10, label='This Work (K₂LiScH₆)')

# Draw the "Ideal Window" (Hypothetical)
plt.axhspan(-0.5, -0.2, color='green', alpha=0.1, label='Ideal Desorption Window')

# Annotations
plt.text(0.75, -0.45, "Too Stable\n(High T_des)", color='blue', fontweight='bold')
plt.text(1.55, -0.15, "Unstable\n(Won't form)", color='red', fontweight='bold')
plt.text(1.38, K2LiScH6_ENERGY + 0.05, "K₂LiScH₆\n(Balanced)",
         horizontalalignment='center', fontweight='bold', color='darkred')

plt.title("Design Strategy: Tuning Stability via Cation Size", fontsize=16)
plt.xlabel("A-Site Ionic Radius (Å)", fontsize=14)
plt.ylabel("Formation Energy (eV/atom)", fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ==========================================
# 4. PLOT 2: THE "SCANDIUM" HEATMAP
# ==========================================
# We parse B and B' again to show Sc stability
def parse_B_sites(comp_str):
    parts = comp_str.split()
    bs = []
    for p in parts:
        ele = re.sub(r'[0-9]+', '', p)
        cnt = re.sub(r'[A-Za-z]+', '', p)
        if ele not in ['H'] and cnt == '1': # Assuming B sites have count 1
            bs.append(ele)
    bs.sort()
    return bs[0] if len(bs) > 0 else None, bs[1] if len(bs) > 1 else None

df[['B', 'B_prime']] = df['composition'].apply(lambda x: pd.Series(parse_B_sites(x)))
heatmap_data = df.pivot_table(index='B', columns='B_prime', values='Predicted_E_form', aggfunc='mean')

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, cmap='RdYlGn_r', linewidths=1, annot=False)

# Highlight where Sc is
# (You can manually add a box in PowerPoint later, or use this text)
plt.title("Chemical Space: Scandium (Sc) Improves Stability", fontsize=16)
plt.xlabel("B' Site Element", fontsize=14)
plt.ylabel("B Site Element", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error
from pymatgen.core import Composition

print("=========================================================")
print("🧐 JOURNAL VALIDATION: DOMAIN-SPECIFIC ERROR ANALYSIS")
print("=========================================================")

# ---------------------------------------------------------
# 1. INTEGRITY CHECK (Crucial for Reviewers)
# ---------------------------------------------------------
# Ensure that the test set features (X_te) align with the original database (df_mp)
# so we can retrieve the correct chemical formulas.
if not set(X_te.index).issubset(set(df_mp.index)):
    raise ValueError("❌ CRITICAL ERROR: Index mismatch. 'X_te' indices are not in 'df_mp'. Cannot retrieve formulas.")
else:
    print("✅ Integrity Check Passed: Test indices align with source data.")

# ---------------------------------------------------------
# 2. DEFINE ROBUST HYDRIDE FILTER
# ---------------------------------------------------------
def is_true_hydride(comp_input):
    """
    Robustly checks if a material contains Hydrogen (H).
    - Prevents false positives from Hafnium (Hf), Mercury (Hg), Holmium (Ho), etc.
    - Handles both Pymatgen Composition objects and string formulas.
    """
    try:
        # Scenario A: Input is already a Pymatgen Composition object
        return "H" in [e.symbol for e in comp_input.elements]
    except AttributeError:
        # Scenario B: Input is a string (e.g., "LiH"), convert first
        try:
            return "H" in [e.symbol for e in Composition(str(comp_input)).elements]
        except:
            return False # Fallback for malformed data

# ---------------------------------------------------------
# 3. IDENTIFY HYDRIDES IN TEST SET
# ---------------------------------------------------------
print("🔍 Scanning test set for hydrides...")

# Retrieve compositions using the safe index
test_indices = X_te.index
test_compositions = df_mp.loc[test_indices, 'composition']

# Apply the filter
mask_hydride = test_compositions.apply(is_true_hydride)
num_hydrides = mask_hydride.sum()
total_test = len(X_te)

print(f"   -> Found {num_hydrides} Hydrides out of {total_test} test samples ({num_hydrides/total_test:.1%}).")

# ---------------------------------------------------------
# 4. CALCULATE DOMAIN-SPECIFIC METRICS
# ---------------------------------------------------------
if num_hydrides > 0:
    # Slice Actual Values (Pandas Series uses Index Alignment)
    y_actual_hydride = y_te[mask_hydride]

    # Slice Predicted Values (Numpy Array uses Boolean Mask)
    # Note: .values ensures we don't trigger an index mismatch error
    y_pred_hydride = y_pred_test[mask_hydride.values]

    # Calculate MAE
    mae_hydride = mean_absolute_error(y_actual_hydride, y_pred_hydride)
    mae_global = mean_absolute_error(y_te, y_pred_test)

    # Calculate Bias Ratio (The "Reviewer Metric")
    bias_ratio = mae_hydride / mae_global

    print("\n📊 FINAL PERFORMANCE REPORT")
    print("-" * 30)
    print(f"   Global MAE (All Materials):  {mae_global:.4f} eV/atom")
    print(f"   Hydride MAE (Target Class):  {mae_hydride:.4f} eV/atom")
    print(f"   Bias Ratio:                  {bias_ratio:.2f}x")
    print("-" * 30)

    # ---------------------------------------------------------
    # 5. AUTOMATED VERDICT FOR MANUSCRIPT
    # ---------------------------------------------------------
    if mae_hydride < 0.20 and bias_ratio < 1.1:
        print("✅ PUBLICATION READY:")
        print("   The model generalizes robustly. It performs as well (or better) on the minority")
        print("   hydride class as it does on the majority oxide class.")
        print(f"   -> STAT TO CITE: 'Hydride MAE of {mae_hydride:.3f} eV/atom (Bias Ratio: {bias_ratio:.2f})'")
    elif mae_hydride < 0.20:
        print("⚠️ CAUTION:")
        print(f"   Absolute error is low ({mae_hydride:.3f}), but bias is visible ({bias_ratio:.2f}x).")
        print("   -> Action: Report the absolute MAE, but acknowledge the slight difference.")
    else:
        print("❌ REVISION REQUIRED:")
        print(f"   Model fails on hydrides (MAE: {mae_hydride:.3f}). Do not publish this result.")
        print("   -> Action: Retrain using 'sample_weight' to balance the classes.")

else:
    print("⚠️ WARNING: No hydrides found in the test set. Check your data filtering.")

In [ ]:
# ==============================================================================
# 6. Q1 JOURNAL VALIDATION: PLOTS & TUNING (Run this after Step 4)
# ==============================================================================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import learning_curve, validation_curve, GridSearchCV
from sklearn.ensemble import HistGradientBoostingRegressor

print("\n📊 Generating Validation Figures for Manuscript...")
print("(This validates that your dataset size is sufficient and model is tuned.)")

# Use a professional style for the plots
plt.style.use('seaborn-v0_8-whitegrid')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# ------------------------------------------------------------------------------
# A. LEARNING CURVE (The "Data Sufficiency" Proof)
# ------------------------------------------------------------------------------
print("   1/3 Calculating Learning Curve (Step-by-step training)...")

# We use the training set (X_tr) only, to keep the test set pure.
train_sizes, train_scores, test_scores = learning_curve(
    HistGradientBoostingRegressor(random_state=42, max_iter=200),
    X_tr, y_tr,
    cv=5,  # 5-Fold Cross-Validation (Gold Standard)
    n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 5), # Check at 10%, 32%, 55%, 77%, 100%
    scoring='neg_mean_absolute_error'
)

# Calculate stats
train_mean = -np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
test_mean = -np.mean(test_scores, axis=1)
test_std = np.std(test_scores, axis=1)

# Plot Learning Curve
ax1.plot(train_sizes, train_mean, 'o-', color="#d62728", label="Training Error")
ax1.plot(train_sizes, test_mean, 'o-', color="#2ca02c", label="Cross-Validation Error")
ax1.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color="#d62728")
ax1.fill_between(train_sizes, test_mean - test_std, test_mean + test_std, alpha=0.1, color="#2ca02c")

ax1.set_title("Learning Curve: Data Sufficiency", fontsize=14, fontweight='bold')
ax1.set_xlabel("Training Samples", fontsize=12)
ax1.set_ylabel("MAE (eV/atom)", fontsize=12)
ax1.legend(loc="best")

# ------------------------------------------------------------------------------
# B. HYPERPARAMETER TUNING PLOT (The "Optimization" Proof)
# ------------------------------------------------------------------------------
print("   2/3 Calculating Validation Curve for Learning Rate...")

# We vary the most important parameter: Learning Rate
param_range = [0.01, 0.05, 0.1, 0.2, 0.3]
train_scores_vc, test_scores_vc = validation_curve(
    HistGradientBoostingRegressor(random_state=42, max_iter=200),
    X_tr, y_tr,
    param_name="learning_rate",
    param_range=param_range,
    cv=5,
    scoring="neg_mean_absolute_error",
    n_jobs=-1
)

# Calculate stats
train_mean_vc = -np.mean(train_scores_vc, axis=1)
train_std_vc = np.std(train_scores_vc, axis=1)
test_mean_vc = -np.mean(test_scores_vc, axis=1)
test_std_vc = np.std(test_scores_vc, axis=1)

# Plot Validation Curve
ax2.plot(param_range, train_mean_vc, 'o-', color="#1f77b4", label="Training Error")
ax2.plot(param_range, test_mean_vc, 'o-', color="#ff7f0e", label="Validation Error")
ax2.fill_between(param_range, train_mean_vc - train_std_vc, train_mean_vc + train_std_vc, alpha=0.1, color="#1f77b4")
ax2.fill_between(param_range, test_mean_vc - test_std_vc, test_mean_vc + test_std_vc, alpha=0.1, color="#ff7f0e")

ax2.set_title("Hyperparameter Tuning: Learning Rate", fontsize=14, fontweight='bold')
ax2.set_xlabel("Learning Rate", fontsize=12)
ax2.set_ylabel("MAE (eV/atom)", fontsize=12)
ax2.legend(loc="best")

plt.tight_layout()
plt.savefig("ML_Validation_Plots_Q1.png", dpi=300)
plt.show()
print("   ✅ Plots saved as 'ML_Validation_Plots_Q1.png'")

# ------------------------------------------------------------------------------
# C. GENERATE TABLE FOR SUPPLEMENTARY INFORMATION
# ------------------------------------------------------------------------------
print("   3/3 Running Grid Search to generate Table S4...")

param_grid = {
    'learning_rate': [0.05, 0.1, 0.2],
    'max_iter': [200, 300],
    'max_leaf_nodes': [31, 63],
    'l2_regularization': [0, 0.1]
}

grid = GridSearchCV(
    HistGradientBoostingRegressor(random_state=42),
    param_grid, cv=3, scoring='neg_mean_absolute_error', n_jobs=-1
)
grid.fit(X_tr, y_tr)

print("\n" + "="*50)
print("📄 COPY THIS TABLE INTO YOUR SUPPLEMENTARY INFO")
print("="*50)
print("| Hyperparameter | Optimized Value |")
print("| :--- | :--- |")
for param, value in grid.best_params_.items():
    print(f"| {param} | **{value}** |")
print("="*50)

Adding Li on the B site

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import re

#
# 1. USER INPUT (CRITICAL STEP)
#

K2LiScH6_ENERGY = -0.328547423  #replaceable with your energy by dft

print(f"o Highlighting K2LiScH6 at {K2LiScH6_ENERGY} eV/atom")


# 2. LOAD & PREPARE DATA



df = pd.read_excel("/content/A2BBpH6_Golden_Candidates.csv")


#df = pd.read_csv("A2BBpH6_Final_Predictions_MP_Only.csv") #without formation energy

#calculation of the radius
# Define Ionic Radii (Shannon Radii in Angstroms)
radii = {
    'Li': 0.76, 'Mg': 0.72, 'Na': 1.02, 'Ca': 1.00,
    'Sc': 0.745, 'Ti': 0.605, 'V': 0.54, 'Cr': 0.615,
    'Mn': 0.83, 'Fe': 0.61, 'Co': 0.65, 'Ni': 0.69,
    'Cu': 0.73, 'Zn': 0.74, 'Al': 0.535, 'K': 1.38,
    'Rb': 1.52, 'Cs': 1.67
}

# For A site
def get_A_site(formula):
    # Simple regex to find the first element (A-site)
    match = re.match(r"([A-Z][a-z]?)", formula)
    return match.group(1) if match else None

df['A_site'] = df['formula'].apply(get_A_site)
df['Radius'] = df['A_site'].map(radii)


# 3. PLOT 1: THE SCATTER PLOT "GOLDILOCKS"

plt.figure(figsize=(12, 7))

# Plot the ML Candidates (Background context)
sns.scatterplot(data=df, x='Radius', y='Predicted_E_form',
                hue='A_site', palette='viridis', alpha=0.6, s=80, edgecolor='k')

# Highlight K2LiScH6 (The Star)
# K radius = 1.38 A
plt.scatter([1.38], [K2LiScH6_ENERGY], color='red', s=400, marker='o',
            edgecolor='black', zorder=10, label='This Work (K₂LiScH₆)')

# Draw the "Ideal Window" (Hypothetical)
plt.axhspan(-0.5, -0.2, color='green', alpha=0.1, label='Ideal Desorption Window')

# Annotations
plt.text(0.75, -0.45, "Too Stable\n(High T_des)", color='blue', fontweight='bold')
plt.text(1.55, -0.15, "Unstable\n(Won't form)", color='red', fontweight='bold')
plt.text(1.38, K2LiScH6_ENERGY + 0.05, "K₂LiScH₆\n(Balanced)",
         horizontalalignment='center', fontweight='bold', color='darkred')

plt.title("Design Strategy: Tuning Stability via Cation Size", fontsize=16)
plt.xlabel("A-Site Ionic Radius (Å)", fontsize=14)
plt.ylabel("Formation Energy (eV/atom)", fontsize=14)
plt.legend(loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


# 4. PLOT 2: THE "SCANDIUM" HEATMAP

# We parse B and B' again to show Sc stability
def parse_B_sites(comp_str):
    parts = comp_str.split()
    bs = []
    for p in parts:
        ele = re.sub(r'[0-9]+', '', p)
        cnt = re.sub(r'[A-Za-z]+', '', p)
        if ele not in ['H'] and cnt == '1': # Assuming B sites have count 1
            bs.append(ele)
    bs.sort()
    return bs[0] if len(bs) > 0 else None, bs[1] if len(bs) > 1 else None

df[['B', 'B_prime']] = df['composition'].apply(lambda x: pd.Series(parse_B_sites(x)))
heatmap_data = df.pivot_table(index='B', columns='B_prime', values='Predicted_E_form', aggfunc='mean')

plt.figure(figsize=(10, 8))
sns.heatmap(heatmap_data, cmap='RdYlGn_r', linewidths=1, annot=False)

# Highlight where Sc is
# (You can manually add a box in PowerPoint later, or use this text)
plt.title("Chemical Space: Scandium (Sc) Improves Stability", fontsize=16)
plt.xlabel("B' Site Element", fontsize=14)
plt.ylabel("B Site Element", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# REVISION CELL W1 — STRATIFIED HYDRIDE PREDICTION ERROR
# ----------------------------------------------------------------------------
# Purpose: replace the single "Hydride MAE" with a chemistry-aware breakdown
# across binary, ternary, complex, and oxyhydride subclasses, so that the
# performance on the actual screening target (complex hydrides) is visible.
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error
from pymatgen.core import Composition, Element

print("="*72)
print("W1: STRATIFIED HYDRIDE PERFORMANCE BREAKDOWN")
print("="*72)

def classify_hydride_subclass(comp):
    """Return one of: not_hydride, binary_hydride, ternary_metal_H,
    complex_hydride, oxyhydride, organic_or_other, other_hydride.
    'complex_hydride' is the elpasolite-relevant class."""
    try:
        elements = set(e.symbol for e in comp.elements)
    except AttributeError:
        elements = set(e.symbol for e in Composition(str(comp)).elements)
    if "H" not in elements:
        return "not_hydride"
    non_H = elements - {"H"}
    if any(e in elements for e in ["C", "N"]):
        return "organic_or_other"
    if "O" in elements and any(
        Element(e).is_metal or Element(e).is_metalloid
        for e in non_H if e != "O"):
        return "oxyhydride"
    n_metals = sum(1 for e in non_H
                   if Element(e).is_metal or Element(e).is_metalloid)
    if len(non_H) == 1 and n_metals == 1:
        return "binary_hydride"
    if len(non_H) == 2 and n_metals == 2:
        return "ternary_metal_H"
    if len(non_H) >= 3 and n_metals >= 3:
        return "complex_hydride"
    return "other_hydride"

# Apply classifier to the held-out test set
test_comps = df_mp.loc[X_te.index, "composition"]
hyd_class  = test_comps.apply(classify_hydride_subclass)
print("\nClass counts in held-out test set:")
print(hyd_class.value_counts().to_string())

def metrics_for(mask_series):
    n = int(mask_series.sum())
    if n == 0: return None
    m = mask_series.values
    y_t = y_te.values[m]; y_p = y_pred_test[m]
    return {
        "N": n,
        "MAE_eV_atom":           float(mean_absolute_error(y_t, y_p)),
        "RMSE_eV_atom":          float(np.sqrt(mean_squared_error(y_t, y_p))),
        "MeanSignedError_eV_atom": float(np.mean(y_p - y_t)),
        "MedianAbsErr_eV_atom":  float(np.median(np.abs(y_p - y_t))),
        "P95AbsErr_eV_atom":     float(np.percentile(np.abs(y_p - y_t), 95)),
    }

CLASS_ORDER = ["not_hydride", "binary_hydride", "ternary_metal_H",
               "complex_hydride", "oxyhydride", "organic_or_other", "other_hydride"]
rows = []
for cls in CLASS_ORDER:
    res = metrics_for(hyd_class == cls)
    if res is not None:
        res["subclass"] = cls; rows.append(res)
overall = metrics_for(pd.Series(True, index=hyd_class.index))
overall["subclass"] = "OVERALL_test_set"; rows.append(overall)
hyd_only = metrics_for((hyd_class != "not_hydride") &
                       (hyd_class != "organic_or_other"))
if hyd_only is not None:
    hyd_only["subclass"] = "ALL_metal_hydrides"; rows.append(hyd_only)

W1_table = pd.DataFrame(rows)[
    ["subclass","N","MAE_eV_atom","RMSE_eV_atom",
     "MeanSignedError_eV_atom","MedianAbsErr_eV_atom","P95AbsErr_eV_atom"]
]
print("\nStratified performance table:")
print(W1_table.to_string(index=False))
W1_table.to_csv("W1_hydride_subclass_performance.csv", index=False)
print("\n>> Saved: W1_hydride_subclass_performance.csv")

In [ ]:
# ============================================================================
# REVISION CELL W2 — LOCO CROSS-VALIDATION ON METAL HYDRIDES
# ----------------------------------------------------------------------------
# Purpose: quantify the model's prospective error on the chemistry class
# closest to the screening target. Trains 5 fresh models, each excluding
# a held-out fold of ternary+ metal hydrides from the full MP training set.
# Runtime: ~1.5-3 hours on Colab CPU; run once and save the result.
# ============================================================================
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("="*72)
print("W2: LOCO-CV ON HYDRIDE SUBDOMAIN")
print("="*72)

# Reuse the classifier from W1 (defined in the previous cell)
all_class    = df_mp["composition"].apply(classify_hydride_subclass)
target_mask  = all_class.isin(["ternary_metal_H", "complex_hydride"])
target_idx_full = df_mp.index[target_mask].tolist()
print(f"\nTernary+ metal hydrides found in MP: {len(target_idx_full)}")

# Subsample if very large (compute control). 500 is plenty for stable percentiles.
MAX_LOCO = 500
rng = np.random.default_rng(42)
if len(target_idx_full) > MAX_LOCO:
    target_idx = sorted(rng.choice(target_idx_full, size=MAX_LOCO,
                                   replace=False).tolist())
    print(f"Random subsample to N = {MAX_LOCO} for LOCO (seed 42).")
else:
    target_idx = list(target_idx_full)

if len(target_idx) < 20:
    raise RuntimeError(
        f"Only {len(target_idx)} hydrides available — LOCO would be unreliable. "
        "Either relax the classifier or report only W1 + W3."
    )

# Use the same hyperparameters as the production model
HGBR_KW = dict(random_state=42, max_iter=300,
               learning_rate=0.1, max_leaf_nodes=63, l2_regularization=0.0)

K = 5
kf = KFold(n_splits=K, shuffle=True, random_state=42)
target_arr = np.array(target_idx)
loco_preds = np.full(len(target_idx), np.nan)
loco_truth = np.full(len(target_idx), np.nan)

for f, (sub_tr, sub_te) in enumerate(kf.split(target_arr)):
    held_out     = set(target_arr[sub_te].tolist())
    train_mp_idx = [i for i in df_mp.index if i not in held_out]
    test_mp_idx  = list(target_arr[sub_te])
    m = HistGradientBoostingRegressor(**HGBR_KW)
    m.fit(X.loc[train_mp_idx], y.loc[train_mp_idx])
    loco_preds[sub_te] = m.predict(X.loc[test_mp_idx])
    loco_truth[sub_te] = y.loc[test_mp_idx].values
    print(f"  Fold {f+1}/{K}: trained on {len(train_mp_idx):>6d}, "
          f"predicted on {len(test_mp_idx):>4d}")

loco_residuals = loco_preds - loco_truth
abs_res        = np.abs(loco_residuals)

W2_summary = {
    "N_hydrides_used":  len(target_idx),
    "LOCO_MAE":         float(abs_res.mean()),
    "LOCO_RMSE":        float(np.sqrt(np.mean(loco_residuals**2))),
    "LOCO_MedianAE":    float(np.median(abs_res)),
    "LOCO_MeanSigned":  float(loco_residuals.mean()),
    "LOCO_P90_halfw":   float(np.percentile(abs_res, 90)),
    "LOCO_P95_halfw":   float(np.percentile(abs_res, 95)),
}
print("\nLOCO-CV summary:")
for k, v in W2_summary.items():
    print(f"  {k:>20s} = {v:.4f}" if isinstance(v, float) else f"  {k:>20s} = {v}")

# Persist for downstream cells
pd.DataFrame({
    "df_mp_index":  target_idx,
    "formula":      [str(df_mp.loc[i, "composition"].reduced_formula) for i in target_idx],
    "y_true":       loco_truth,
    "y_pred":       loco_preds,
    "abs_residual": abs_res,
}).to_csv("W2_loco_residuals.csv", index=False)
np.save("W2_loco_residuals.npy", loco_residuals)
print("\n>> Saved: W2_loco_residuals.csv  and  W2_loco_residuals.npy")

In [ ]:
# ============================================================================
# REVISION CELL W4 — HYDRIDE-LOCALISED SPLIT-CONFORMAL PREDICTION INTERVALS
# ----------------------------------------------------------------------------
# Purpose: turn the LOCO residuals from W2 into per-candidate prediction
# intervals with calibrated empirical coverage. Apply to the full screened
# candidate list and re-rank.
# ============================================================================
import os
import numpy as np
import pandas as pd
from sklearn.utils import resample

print("="*72)
print("W4: HYDRIDE-LOCALISED CONFORMAL PREDICTION INTERVALS")
print("="*72)

# Calibration from W2 ----------------------------------------------------------
HW90 = float(np.percentile(np.abs(loco_residuals), 90))
HW95 = float(np.percentile(np.abs(loco_residuals), 95))
loco_mae = float(np.mean(np.abs(loco_residuals)))

# Bootstrap CI on HW90 itself (to honestly report the half-width's own uncertainty)
B_BOOT = 2000
boot_HW90 = np.array([
    np.percentile(np.abs(resample(loco_residuals, replace=True)), 90)
    for _ in range(B_BOOT)
])
HW90_lo, HW90_hi = float(np.percentile(boot_HW90, 2.5)), float(np.percentile(boot_HW90, 97.5))

print(f"\nCalibrated from N = {len(loco_residuals)} LOCO hydride residuals:")
print(f"  90% conformal half-width: {HW90:.3f} eV/atom "
      f"(bootstrap 95% CI [{HW90_lo:.3f}, {HW90_hi:.3f}])")
print(f"  95% conformal half-width: {HW95:.3f} eV/atom")

# Locate the screening file (try both Colab and local paths) -------------------
SCREEN_PATH_CANDIDATES = [
    "/content/Final ML screening.xlsx",
    "/content/Final_ML_screening.xlsx",
    "Final_ML_screening.xlsx",
    "Final ML screening.xlsx",
    "/content/A2BBpH6_Golden_Candidates.csv",
    "A2BBpH6_Golden_Candidates.csv",
]
src = next((p for p in SCREEN_PATH_CANDIDATES if os.path.exists(p)), None)
if src is None:
    raise FileNotFoundError(
        "Could not locate the screening file. Looked at: "
        + ", ".join(SCREEN_PATH_CANDIDATES) +
        ". Edit SCREEN_PATH_CANDIDATES with the correct path."
    )
print(f"\nScreening file: {src}")

if src.endswith(".xlsx"):
    df_screen = pd.read_excel(src)
else:
    df_screen = pd.read_csv(src)

assert "Predicted_E_form" in df_screen.columns, (
    f"Need column 'Predicted_E_form' in screening file; got {df_screen.columns.tolist()}"
)
assert "formula" in df_screen.columns, (
    f"Need column 'formula' in screening file; got {df_screen.columns.tolist()}"
)

CUTOFF = -0.15

df_screen["lower90_eV_atom"]    = df_screen["Predicted_E_form"] - HW90
df_screen["upper90_eV_atom"]    = df_screen["Predicted_E_form"] + HW90
df_screen["lower95_eV_atom"]    = df_screen["Predicted_E_form"] - HW95
df_screen["upper95_eV_atom"]    = df_screen["Predicted_E_form"] + HW95
df_screen["passes_pointwise"]   = df_screen["Predicted_E_form"] < CUTOFF
df_screen["passes_with_90upper"] = df_screen["upper90_eV_atom"] < CUTOFF
df_screen["passes_with_95upper"] = df_screen["upper95_eV_atom"] < CUTOFF
df_screen = df_screen.sort_values("Predicted_E_form").reset_index(drop=True)
df_screen.insert(0, "rank", np.arange(1, len(df_screen)+1))

n_total   = len(df_screen)
n_pass_pt = int(df_screen["passes_pointwise"].sum())
n_pass_90 = int(df_screen["passes_with_90upper"].sum())
n_pass_95 = int(df_screen["passes_with_95upper"].sum())
print(f"\nOf {n_total} candidates:")
print(f"  Passes pointwise (no uncertainty):           {n_pass_pt}")
print(f"  Robust against 90% conformal upper bound:    {n_pass_90}  "
      f"({100*n_pass_90/n_total:.0f}%)")
print(f"  Robust against 95% conformal upper bound:    {n_pass_95}  "
      f"({100*n_pass_95/n_total:.0f}%)")

# Locate K2LiScH6 specifically
mask_KL = df_screen["formula"].astype(str).str.replace(" ", "").str.contains("K2LiScH6")
if mask_KL.sum() == 0:
    print("\nWARNING: K2LiScH6 not found in the screening table.")
else:
    rec = df_screen.loc[mask_KL].iloc[0]
    print(f"\nK2LiScH6 (rank {int(rec['rank'])}/{n_total}):")
    print(f"  Predicted E_form:        {rec['Predicted_E_form']:+.3f} eV/atom")
    print(f"  90% conformal interval:  [{rec['lower90_eV_atom']:+.3f}, "
          f"{rec['upper90_eV_atom']:+.3f}]")
    print(f"  95% conformal interval:  [{rec['lower95_eV_atom']:+.3f}, "
          f"{rec['upper95_eV_atom']:+.3f}]")
    print(f"  Robust to 90% upper?     "
          f"{'YES' if rec['passes_with_90upper'] else 'NO (interval straddles cutoff)'}")
    print(f"  Robust to 95% upper?     "
          f"{'YES' if rec['passes_with_95upper'] else 'NO (interval straddles cutoff)'}")

df_screen.to_excel("W4_screening_with_conformal_intervals.xlsx", index=False)
print("\n>> Saved: W4_screening_with_conformal_intervals.xlsx")

In [ ]:
# ============================================================================
# REVISION CELL W5 (FINAL) — DOMAIN-SPECIFIC VALIDATION OF THE HGBR MODEL
# ----------------------------------------------------------------------------
# Two-panel figure for the revised SI:
#   (a) W1 — stratified hydride MAE: shows that prediction error depends on
#       the hydride sub-class. The complex-hydride class (red) is the closest
#       chemistry analogue to the A2BB'H6 screening target.
#   (b) W2 — LOCO-CV residual distribution on hydrides: empirical 90% / 95%
#       prediction intervals calibrated on held-out ternary+ metal hydrides.
#
# Requires the following objects in memory:
#   - W1_table        (from Cell W1)
#   - loco_residuals  (from Cell W2)
# ============================================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import matplotlib.patches as mpatches

# ----- pre-flight checks ----------------------------------------------------
missing = [n for n, ok in [
    ("W1_table",        "W1_table"       in dir()),
    ("loco_residuals",  "loco_residuals" in dir()),
] if not ok]
if missing:
    raise RuntimeError(
        "Missing prerequisite objects: " + ", ".join(missing) +
        ". Run cells W1 and W2 before this cell."
    )

# ----- journal-style rcParams ----------------------------------------------
mpl.rcParams.update({
    "font.family":          "DejaVu Sans",
    "font.size":            10.5,
    "axes.linewidth":       1.1,
    "axes.labelsize":       11.5,
    "axes.titlesize":       12,
    "axes.titleweight":     "bold",
    "xtick.major.size":     4.5,
    "ytick.major.size":     4.5,
    "xtick.major.width":    1.0,
    "ytick.major.width":    1.0,
    "legend.frameon":       False,
    "legend.fontsize":      9.5,
    "figure.dpi":           120,
    "savefig.dpi":          600,
    "savefig.bbox":         "tight",
    "mathtext.default":     "regular",
})

# Two-panel layout side by side, with explicit margins so nothing crowds
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13.0, 5.6))
plt.subplots_adjust(wspace=0.28, top=0.88, bottom=0.13, left=0.07, right=0.97)

# =========================================================================
# PANEL (a): stratified hydride MAE
# =========================================================================
PLOT_CLASSES = ["not_hydride", "binary_hydride", "ternary_metal_H",
                "complex_hydride", "oxyhydride"]
LABELS = {
    "not_hydride":      "Non-hydride",
    "binary_hydride":   "Binary",
    "ternary_metal_H":  "Ternary\nmetal",
    "complex_hydride":  "Complex\n(target)",
    "oxyhydride":       "Oxyhydride",
}
sub    = W1_table.set_index("subclass").reindex(PLOT_CLASSES).dropna()
xpos   = np.arange(len(sub))
err_up = (sub["P95AbsErr_eV_atom"] - sub["MAE_eV_atom"]).values

# Error bars first so the bars overlap their bases cleanly
ax1.errorbar(xpos, sub["MAE_eV_atom"],
             yerr=[np.zeros_like(err_up), err_up],
             fmt="none", ecolor="#222222",
             elinewidth=1.6, capsize=8, capthick=1.6, zorder=3)

# Color the target class red, others blue
bar_colors = ["#4C72B0"] * len(sub)
for i, name in enumerate(sub.index):
    if name == "complex_hydride":
        bar_colors[i] = "#C44E52"
bars = ax1.bar(xpos, sub["MAE_eV_atom"], color=bar_colors,
               edgecolor="black", linewidth=1.0, width=0.62, zorder=2)

# N labels inside each bar (vertically centred on each MAE)
for i, n in enumerate(sub["N"]):
    ax1.text(xpos[i], sub["MAE_eV_atom"].iloc[i] * 0.5,
             f"N={int(n)}", ha="center", va="center",
             fontsize=9, color="white", fontweight="bold", zorder=4)

# Reported global MAE — make the line and its label CLEARLY visible
ax1.axhline(0.151, ls="--", color="#444444", lw=1.3, zorder=1)
ax1.text(-0.32, 0.158, "Reported global MAE = 0.151 eV/atom",
         ha="left", va="bottom", fontsize=10, color="#333333",
         fontweight="bold", style="italic",
         bbox=dict(boxstyle="round,pad=0.25", fc="white",
                   ec="#bbbbbb", lw=0.6))

# Error-bar key
ax1.text(0.985, 0.965,
         r"Error bars: 95$^{\mathrm{th}}$ percentile of |residual|",
         transform=ax1.transAxes, ha="right", va="top",
         fontsize=9, color="#333333", style="italic",
         bbox=dict(boxstyle="round,pad=0.3", fc="white",
                   ec="#bbbbbb", lw=0.6))

ax1.set_xticks(xpos)
ax1.set_xticklabels([LABELS[c] for c in sub.index], fontsize=10)
ax1.set_ylabel("MAE (eV/atom)", fontsize=11.5)
ax1.set_title("(a) Domain-stratified prediction error", loc="left", pad=10)
ax1.set_ylim(0, sub["P95AbsErr_eV_atom"].max() * 1.22)
ax1.set_xlim(-0.55, len(sub) - 0.45)
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", ls=":", color="#cccccc", lw=0.6, zorder=0)

# =========================================================================
# PANEL (b): hydride LOCO-CV residual distribution
# =========================================================================
HW90      = float(np.percentile(np.abs(loco_residuals), 90))
HW95      = float(np.percentile(np.abs(loco_residuals), 95))
loco_mae  = float(np.mean(np.abs(loco_residuals)))
loco_bias = float(loco_residuals.mean())

# Histogram first
counts, bins, _ = ax2.hist(
    loco_residuals, bins=40, color="#55A868",
    edgecolor="black", linewidth=0.5, alpha=0.85, zorder=2)

# Reserve top 22% of the y-axis for the legend and stats box (no overlap)
ymax = counts.max() * 1.30
ax2.set_ylim(0, ymax)

# Interval bands only fill the lower 78% of the panel
ax2.axvspan(-HW95, -HW90, ymin=0, ymax=0.78,
            color="#C44E52", alpha=0.13, zorder=1)
ax2.axvspan( HW90,  HW95, ymin=0, ymax=0.78,
            color="#C44E52", alpha=0.13, zorder=1)
ax2.axvspan(-HW90,  HW90, ymin=0, ymax=0.78,
            color="#4C72B0", alpha=0.18, zorder=1)

# Zero line + dashed markers at ±HW90 and ±HW95 so the edges are explicit
ax2.axvline(0, color="black", lw=1.0, zorder=3)
for x in [-HW95, -HW90, HW90, HW95]:
    ax2.axvline(x, color="#888888", ls=":", lw=0.9, zorder=2)

ax2.set_xlabel("LOCO residual: predicted − reference (eV/atom)", fontsize=11.5)
ax2.set_ylabel("Count", fontsize=11.5)
ax2.set_title(f"(b) Hydride LOCO-CV residuals (N = {len(loco_residuals)})",
              loc="left", pad=10)
ax2.spines[["top", "right"]].set_visible(False)
ax2.grid(axis="y", ls=":", color="#cccccc", lw=0.6, zorder=0)

# Stats box at top-left (in the reserved label zone)
ax2.text(0.025, 0.965,
         f"MAE = {loco_mae:.3f} eV/atom\n"
         f"Mean signed = {loco_bias:+.3f} eV/atom",
         transform=ax2.transAxes, va="top", ha="left", fontsize=10,
         bbox=dict(boxstyle="round,pad=0.35", fc="white",
                   ec="#888888", lw=0.7))

# Interval legend at top-right (also in the reserved label zone)
proxy90 = mpatches.Patch(color="#4C72B0", alpha=0.45,
                         label=f"90% interval (±{HW90:.2f})")
proxy95 = mpatches.Patch(color="#C44E52", alpha=0.35,
                         label=f"95% interval (±{HW95:.2f})")
leg = ax2.legend(handles=[proxy90, proxy95], loc="upper right",
                 frameon=True, framealpha=0.95, edgecolor="#888888",
                 fontsize=9.5, bbox_to_anchor=(0.985, 0.975))
leg.get_frame().set_linewidth(0.7)

# =========================================================================
# Title and save
# =========================================================================
#fig.suptitle("Figure S4. Domain-specific validation of the HGBR screening model.",
            # fontsize=13, fontweight="bold", y=0.98)

plt.savefig("Figure_S4_validation.png", dpi=600, bbox_inches="tight",
            facecolor="white")
plt.savefig("Figure_S4_validation.pdf", bbox_inches="tight", facecolor="white")
plt.show()
print(">> Saved: Figure_S4_validation.png and Figure_S4_validation.pdf")

In [ ]:
print(f"Total rows in df_mp: {len(df_mp)}")
print(f"Unique formulas:     {df_mp['composition'].apply(lambda c: c.reduced_formula).nunique()}")
print(f"Duplicate ratio:     {len(df_mp) / df_mp['composition'].apply(lambda c: c.reduced_formula).nunique():.2f}")

In [ ]:
df_mp["_red_form"] = df_mp["composition"].apply(lambda c: c.reduced_formula)
df_mp = df_mp.sort_values("e_form").drop_duplicates(
    subset="_red_form", keep="first").drop(columns="_red_form").reset_index(drop=True)
print(f"After dedup: {len(df_mp)} rows")